In [ ]:
from googleapiclient.discovery import build
import pandas as pd

# 1. API 키 설정 및 서비스 빌드
api_key = 'YOUR_API_KEY_HERE'
youtube = build('youtube', 'v3', developerKey=api_key)

# 2. 검색 키워드 설정 (질문자님 담당 섹터 중심)
search_keywords = ["고배당주 추천", "SCHD 전망", "KOFR 금리", "파킹통장 ETF", "트럼프 관세 주식"]

def get_youtube_data(keyword, max_results=50):
    # 특정 기간(2019~2025) 설정을 위해 publishedAfter/Before 옵션 활용 가능
    request = youtube.search().list(
        q=keyword,
        part="snippet",
        type="video",
        maxResults=max_results,
        order="viewCount" # 조회수 높은 순으로 수집
    )
    response = request.execute()

    video_data = []
    for item in response['items']:
        video_id = item['id']['videoId']

        # 상세 통계(조회수, 좋아요 등)를 위해 추가 호출
        stats_request = youtube.videos().list(
            part="statistics",
            id=video_id
        )
        stats_response = stats_request.execute()
        stats = stats_response['items'][0]['statistics']

        video_data.append({
            'title': item['snippet']['title'],
            'published_at': item['snippet']['publishedAt'],
            'view_count': stats.get('viewCount', 0),
            'like_count': stats.get('likeCount', 0),
            'comment_count': stats.get('commentCount', 0),
            'video_id': video_id
        })
    return video_data

# 실행 및 저장
all_results = []
for kw in search_keywords:
    all_results.extend(get_youtube_data(kw))

df = pd.DataFrame(all_results)
df.to_csv('youtube_market_sentiment.csv', index=False)
print("수집 완료!")